# Notebook 13 — Investigation Queue Validation and Investigative Usefulness

**Purpose:** Stress-test the **2024 organization-level research queues** produced by Notebook 12 before considering operational use. This notebook performs data-quality and historical coverage audits, ranking sensitivity tests, peer-group diagnostics, and produces a **de-identified review rubric**. It does **not** train a new model, treat anomalies as fraud, or claim an external validation set.

**Inputs:** Notebook 12 primary and limited-data queues, plus the canonical longitudinal features file.  
**Outputs:** `data/processed/notebook13/`. **Run:** top to bottom in the same `irs990` environment. This notebook uses DuckDB, pandas, numpy, and matplotlib.

**Critical caution:** 2023–2024 outcomes have already informed earlier development. These are retrospective diagnostics, **not independent validation of fraud detection**.

## 1. Setup and input contract

Check required files and columns before analysis. Fail explicitly if an upstream schema changes.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'
IN12_PRIMARY = DATA / 'notebook12' / '07_2024_organization_primary_queue.parquet'
IN12_LIMITED = DATA / 'notebook12' / '07_2024_organization_limited_data_queue.parquet'
FEATURES = DATA / 'nccs_longitudinal_features_v1.parquet'
OUT = DATA / 'notebook13'
OUT.mkdir(parents=True, exist_ok=True)
TOP_K = (100, 500, 1000)
for path in [IN12_PRIMARY, IN12_LIMITED, FEATURES]:
    assert path.is_file(), f'Missing prerequisite: {path}'

def sqlpath(path):
    return str(path).replace("'", "''")

def norm_ein(s):
    # Accept EIN-12-3456789, 12-3456789, 123456789, and numeric-looking strings.
    z = s.astype('string').str.strip().str.replace(r'^EIN-', '', regex=True)
    z = z.str.replace('-', '', regex=False).str.replace(r'\.0$', '', regex=True).str.zfill(9)
    return z

def numeric(s):
    return pd.to_numeric(s, errors='coerce').astype('float64').replace([np.inf, -np.inf], np.nan)

def top_set(frame, col, k):
    return set(frame.loc[numeric(frame[col]).le(k).fillna(False), 'ein'])

con = duckdb.connect()
con.execute('SET threads=4')
con.execute("SET memory_limit='6GB'")
con.execute(f"CREATE VIEW features AS SELECT * FROM read_parquet('{sqlpath(FEATURES)}')")
p = pd.read_parquet(IN12_PRIMARY)
l = pd.read_parquet(IN12_LIMITED)
required = {'ein','OBJECTID','tax_year','integration_cohort','peer_group',
            'organization_consensus_rank','organization_supervised_rank',
            'organization_peer_rank','supervised_diversion_score',
            'peer_adverse_6','consensus_score','six_indicator_count'}
assert required.issubset(p.columns), f'Missing primary fields: {sorted(required-set(p.columns))}'
assert required.issubset(l.columns), f'Missing limited fields: {sorted(required-set(l.columns))}'
assert set(p.columns)==set(l.columns), 'Primary and limited schemas differ'
for frame in (p,l):
    frame['ein'] = norm_ein(frame['ein'])
    assert frame.ein.str.fullmatch(r'\d{9}').fillna(False).all(), 'Invalid EIN after normalization'
    assert frame.ein.is_unique
    assert frame.tax_year.eq(2024).all()
assert set(p.ein).isdisjoint(set(l.ein)), 'EIN appears in both queues'
assert p.organization_consensus_rank.notna().all()
assert l.organization_consensus_rank.isna().all()
assert p.integration_cohort.eq('primary_review').all()
assert l.integration_cohort.eq('limited_data').all()
print('Primary:',f'{len(p):,}','Limited:',f'{len(l):,}','Unique total:',f'{len(p)+len(l):,}')
print('Columns:',len(p.columns))

## 2. Historical coverage and integrity audit

**Important upstream check:** Notebook 12 may have joined the historical file with unnormalized `EIN-XX-XXXXXXX` values. That can silently produce empty history. This notebook independently normalizes EINs in DuckDB and compares historical source coverage with the fields exported by Notebook 12. A discrepancy is a **blocking upstream data issue**, not a reason to manufacture history.

The canonical feature table can contain more than one filing per EIN/year. Historical *coverage* counts distinct years; we do not silently pick a conflicting filing to compute trends.

In [ ]:
fcols = set(con.execute('DESCRIBE features').df().column_name)
needed = {'ein','tax_year','OBJECTID','total_revenue','total_assets','total_expenses','total_liabilities','net_income'}
assert needed.issubset(fcols), f'Missing historical columns: {sorted(needed-fcols)}'
all_eins = pd.concat([p[['ein']],l[['ein']]],ignore_index=True)
con.register('queue_eins', all_eins)
# Normalize the historical EIN in SQL before joining. Keep the query restricted to 2022–2024.
hist = con.execute(r"""
WITH h AS (
 SELECT LPAD(REGEXP_REPLACE(REGEXP_REPLACE(TRIM(CAST(ein AS VARCHAR)), '^EIN-', ''), '[-]', '', 'g'), 9, '0') AS ein,
        TRY_CAST(tax_year AS INTEGER) AS tax_year,
        CAST(OBJECTID AS VARCHAR) AS objectid,
        TRY_CAST(total_revenue AS DOUBLE) AS revenue,
        TRY_CAST(total_assets AS DOUBLE) AS assets,
        TRY_CAST(total_expenses AS DOUBLE) AS expenses,
        TRY_CAST(total_liabilities AS DOUBLE) AS liabilities,
        TRY_CAST(net_income AS DOUBLE) AS net_income
 FROM features WHERE tax_year BETWEEN 2022 AND 2024
)
SELECT h.* FROM h INNER JOIN queue_eins q USING (ein)
""").df()
assert hist.ein.str.fullmatch(r'\d{9}').all()
assert hist.tax_year.isin([2022,2023,2024]).all()
# Duplicate EIN/year filings can differ; report them rather than choosing an arbitrary record.
year_counts = hist.groupby(['ein','tax_year'],observed=True).size().rename('source_filings').reset_index()
dups = year_counts.loc[year_counts.source_filings.gt(1)].copy()
source_years = hist.groupby(['ein','tax_year'],observed=True).agg(
    source_revenue_present=('revenue',lambda x: x.notna().any()),
    source_assets_present=('assets',lambda x: x.notna().any()),
    source_expenses_present=('expenses',lambda x: x.notna().any()),
    source_liabilities_present=('liabilities',lambda x: x.notna().any()),
    source_net_income_present=('net_income',lambda x: x.notna().any()),
).reset_index()
coverage = all_eins.copy()
for year in [2022,2023,2024]:
    y = source_years.loc[source_years.tax_year.eq(year)].drop(columns='tax_year').copy()
    y = y.rename(columns={c:f'{c}_{year}' for c in y.columns if c!='ein'})
    y[f'source_has_filing_{year}'] = True
    coverage = coverage.merge(y,on='ein',how='left',validate='one_to_one')
    coverage[f'source_has_filing_{year}'] = coverage[f'source_has_filing_{year}'].fillna(False).astype(bool)
coverage['source_has_all_3_years'] = coverage[[f'source_has_filing_{y}' for y in [2022,2023,2024]]].all(axis=1)
for metric in ['revenue','assets','expenses','liabilities','net_income']:
    for year in [2022,2023,2024]:
        col = f'source_{metric}_present_{year}'
        coverage[col] = coverage[col].fillna(False).astype(bool)
combined = pd.concat([p,l],ignore_index=True)
check = combined[['ein','integration_cohort']].copy()
for metric in ['revenue','assets','expenses','liabilities','net_income']:
    for year in [2022,2023,2024]:
        col = f'{metric}_{year}'
        check[f'export_{metric}_{year}_present'] = numeric(combined[col]).notna().to_numpy() if col in combined else False
check = check.merge(coverage,on='ein',how='left',validate='one_to_one')
rows=[]
for year in [2022,2023,2024]:
    rows.append({'year':year,'organizations_with_source_filing':int(check[f'source_has_filing_{year}'].sum()),
                 'source_revenue_nonmissing':int(check[f'source_revenue_present_{year}'].sum()),
                 'notebook12_revenue_nonmissing':int(check[f'export_revenue_{year}_present'].sum()),
                 'source_assets_nonmissing':int(check[f'source_assets_present_{year}'].sum()),
                 'notebook12_assets_nonmissing':int(check[f'export_assets_{year}_present'].sum())})
history_summary = pd.DataFrame(rows)
display(history_summary)
print('Organizations with source filings in all 3 years:',int(check.source_has_all_3_years.sum()))
print('Duplicate EIN/year groups in historical source:',len(dups))
print('Historical source rows:',len(hist))
history_summary.to_csv(OUT/'02_historical_coverage_summary.csv',index=False)
dups.to_csv(OUT/'02_historical_duplicate_ein_years.csv',index=False)
# Aggregate audit; do not export individual organization identifiers here.
coverage.groupby('source_has_all_3_years').size().rename('organizations').to_csv(OUT/'02_three_year_coverage.csv')
upstream_history_mismatch = any(
    (check[f'source_{metric}_present_{year}'] & ~check[f'export_{metric}_{year}_present']).any()
    for metric in ['revenue','assets','expenses','liabilities','net_income']
    for year in [2022,2023,2024]
)
if upstream_history_mismatch:
    print('WARNING: Notebook 12 exported historical fields are missing despite data existing in source. Fix and rerun Notebook 12 before interpreting historical trends.')
else:
    print('No missing-export historical coverage discrepancy detected.')

## 3. Queue completeness and high-rank data quality

Compare missingness, indicator coverage, and extreme-value prevalence for the top 100/500/1,000 and the full primary queue. Extreme values are **screening flags**, not data errors by definition.

In [ ]:
metrics = [c for c in ['revenue','expenses','assets','liabilities','net_income',
    'expense_revenue_ratio','liabilities_assets_ratio','revenue_growth_3yr',
    'asset_growth_3yr','revenue_volatility_3yr','board_independence'] if c in p.columns]
for c in metrics:
    p[c] = numeric(p[c])
quality_rows=[]
for name, subset in [('all_primary',p)]+[(f'top_{k}',p.loc[numeric(p.organization_consensus_rank).le(k)]) for k in TOP_K]:
    r={'group':name,'organizations':len(subset),
       'mean_six_indicator_count':float(numeric(subset.six_indicator_count).mean()),
       'pct_with_6_indicators':float(numeric(subset.six_indicator_count).eq(6).mean()*100)}
    for c in metrics:
        r[f'{c}_missing_pct'] = float(subset[c].isna().mean()*100)
    quality_rows.append(r)
quality = pd.DataFrame(quality_rows)
display(quality)
quality.to_csv(OUT/'03_queue_completeness.csv',index=False)
# Flag large magnitudes using *full primary* distribution, with no assumption that they are incorrect.
extreme_cols = [c for c in ['revenue','expenses','assets','liabilities','net_income','expense_revenue_ratio','liabilities_assets_ratio'] if c in p]
thresholds={}
for c in extreme_cols:
    x=p[c].abs().dropna()
    thresholds[c] = float(x.quantile(.999)) if len(x) else np.nan
extreme_rows=[]
for name, subset in [('all_primary',p)]+[(f'top_{k}',p.loc[numeric(p.organization_consensus_rank).le(k)]) for k in TOP_K]:
    r={'group':name,'organizations':len(subset)}
    for c in extreme_cols:
        t=thresholds[c]
        r[f'{c}_abs_gt_p999_pct'] = float(subset[c].abs().gt(t).mean()*100) if np.isfinite(t) else np.nan
    extreme_rows.append(r)
extremes=pd.DataFrame(extreme_rows)
display(extremes)
extremes.to_csv(OUT/'03_extreme_value_prevalence.csv',index=False)
(OUT/'03_extreme_thresholds.json').write_text(json.dumps(thresholds,indent=2))

## 4. Ranking sensitivity: weights and top-K stability

Recompute an **alternative percentile-rank blend** from the exported component ranks (supervised and peer) at 25/75, 50/50, and 75/25. Compare each alternative with the **actual Notebook 12 consensus ranking**. This is a sensitivity test, not a reproduction guarantee: the upstream consensus may use different percentile conventions or tie handling. Report any discrepancy instead of silently replacing the official queue.

In [ ]:
rank = p[['ein','organization_consensus_rank','organization_supervised_rank','organization_peer_rank']].copy()
N=len(rank)
rank['supervised_pct'] = (N+1-numeric(rank.organization_supervised_rank))/N
rank['peer_pct'] = (N+1-numeric(rank.organization_peer_rank))/N
assert rank[['supervised_pct','peer_pct']].notna().all().all()
for w in [.25,.50,.75]:
    key=f'blend_{int(w*100)}_{int((1-w)*100)}'
    rank[key+'_score'] = w*rank.supervised_pct+(1-w)*rank.peer_pct
    order=rank.sort_values([key+'_score','ein'],ascending=[False,True],kind='mergesort').index
    rank[key+'_rank']=pd.Series(np.arange(1,N+1),index=order).reindex(rank.index).astype('int64')
ref={k:top_set(rank,'organization_consensus_rank',k) for k in TOP_K}
weight_rows=[]
for w in [.25,.50,.75]:
    key=f'blend_{int(w*100)}_{int((1-w)*100)}'
    for k in TOP_K:
        alt=top_set(rank,key+'_rank',k)
        weight_rows.append({'supervised_weight':w,'top_k':k,
            'overlap_with_notebook12_consensus':len(alt&ref[k]),
            'pct_of_top_k_retained':100*len(alt&ref[k])/k,
            'organizations_replaced':k-len(alt&ref[k])})
weight_sensitivity=pd.DataFrame(weight_rows)
display(weight_sensitivity)
weight_sensitivity.to_csv(OUT/'04_weight_sensitivity.csv',index=False)
# Inspect a specific tie-handling issue without changing any upstream rank.
print('Spearman rank correlations vs Notebook 12 consensus:')
for w in [.25,.50,.75]:
    col=f'blend_{int(w*100)}_{int((1-w)*100)}_rank'
    print(col,round(rank[['organization_consensus_rank',col]].corr(method='spearman').iloc[0,1],4))
fig,ax=plt.subplots(figsize=(8,4))
for w,sub in weight_sensitivity.groupby('supervised_weight'):
    ax.plot(sub.top_k,sub.pct_of_top_k_retained,marker='o',label=f'Supervised weight {w:.0%}')
ax.set(xlabel='Top-K threshold',ylabel='Overlap with Notebook 12 consensus (%)',title='Weight sensitivity of investigation queue')
ax.set_ylim(0,105)
ax.legend()
fig.tight_layout()
fig.savefig(OUT/'04_weight_sensitivity.png',dpi=150)
plt.show()

## 5. Ranking sensitivity: threshold choices and component agreement

Top-K cutoffs are operational choices, not statistical significance thresholds. Measure overlap and the prevalence of the four signal patterns as K changes.

In [ ]:
agreement=[]
for k in [100,250,500,1000,2000,5000]:
    s=top_set(p,'organization_supervised_rank',k)
    a=top_set(p,'organization_peer_rank',k)
    c=top_set(p,'organization_consensus_rank',k)
    agreement.append({'top_k':k,'supervised_peer_overlap':len(s&a),
       'supervised_peer_jaccard':len(s&a)/len(s|a),
       'supervised_consensus_overlap':len(s&c),
       'peer_consensus_overlap':len(a&c),
       'either_component':len(s|a),'both_components':len(s&a)})
agreement=pd.DataFrame(agreement)
display(agreement)
agreement.to_csv(OUT/'05_threshold_agreement.csv',index=False)
assert agreement.loc[agreement.top_k.eq(1000),'supervised_peer_overlap'].iloc[0] >= 0

## 6. Peer-group composition and sensitivity proxy

The existing Notebook 12 queue contains revenue-based peer groups, but not necessarily the full reference distribution or underlying Notebook 10 scoring transformations. We **cannot honestly re-score alternative mission-based or differently binned peer groups** from the exported queue alone. Instead, this section tests how strongly the top ranks concentrate in existing peer groups and calculates a *descriptive proxy* using global rather than within-group percentiles for the six raw metrics. The proxy is **not** a replacement anomaly score; directions are aligned to adverse behavior, and no winsorization or Notebook 10 transformations are replicated.

In [ ]:
groups = p.groupby('peer_group',dropna=False).size().rename('population').reset_index()
for k in TOP_K:
    sub=p.loc[numeric(p.organization_consensus_rank).le(k)].groupby('peer_group',dropna=False).size().rename(f'top{k}')
    groups=groups.merge(sub.reset_index(),on='peer_group',how='left')
    groups[f'top{k}']=groups[f'top{k}'].fillna(0).astype(int)
    groups[f'top{k}_share_pct']=100*groups[f'top{k}']/k
    groups[f'top{k}_population_share_pct']=100*groups.population/len(p)
    groups[f'top{k}_enrichment']=groups[f'top{k}_share_pct']/groups[f'top{k}_population_share_pct'].replace(0,np.nan)
groups=groups.sort_values('population',ascending=False)
display(groups)
groups.to_csv(OUT/'06_peer_group_concentration.csv',index=False)
proxy_directions={'expense_revenue_ratio':1,'liabilities_assets_ratio':1,
                  'revenue_growth_3yr':-1,'asset_growth_3yr':-1,
                  'revenue_volatility_3yr':1,'board_independence':-1}
available=[c for c in proxy_directions if c in p.columns]
assert len(available)>=4, 'Insufficient raw peer metrics for descriptive proxy'
proxy=pd.DataFrame(index=p.index)
for c in available:
    x=numeric(p[c])*proxy_directions[c]
    proxy[c]=x.rank(pct=True,method='average')
p['global_percentile_proxy']=proxy.mean(axis=1,skipna=True)
p['global_proxy_available']=proxy.notna().sum(axis=1)
# Fairer comparison: same >=4 metric coverage policy as the primary cohort.
valid=p.global_proxy_available.ge(4)&p.global_percentile_proxy.notna()
proxy_rank=p.loc[valid,['ein','global_percentile_proxy']].sort_values(
    ['global_percentile_proxy','ein'],ascending=[False,True],kind='mergesort')
proxy_rank['global_proxy_rank']=np.arange(1,len(proxy_rank)+1)
proxy_results=[]
for k in TOP_K:
    orig=top_set(p,'organization_peer_rank',k)
    proxy_top=set(proxy_rank.head(k).ein)
    proxy_results.append({'top_k':k,'peer_vs_global_descriptive_proxy_overlap':len(orig&proxy_top),
                          'proxy_eligible':len(proxy_rank)})
proxy_results=pd.DataFrame(proxy_results)
display(proxy_results)
proxy_results.to_csv(OUT/'06_global_proxy_comparison.csv',index=False)
print('This comparison is a diagnostic proxy only; it does not implement alternative peer-group anomaly scoring.')

## 7. Signal-pattern comparisons and retrospective disclosure limitations

Compare financial and data-coverage profiles for organizations in both top 1,000 lists, only one, or neither. **No fraud label is present in the Notebook 12 exports by design.** Do not infer confirmed fraud or calculate precision/recall from these groups.

In [ ]:
s=top_set(p,'organization_supervised_rank',1000)
a=top_set(p,'organization_peer_rank',1000)
p['validation_signal_pattern']=np.select(
    [p.ein.isin(s&a),p.ein.isin(s-a),p.ein.isin(a-s)],
    ['both','supervised_only','peer_only'],default='neither')
compare_cols=[c for c in ['revenue','assets','liabilities','expenses','net_income',
    'six_indicator_count','expense_revenue_ratio','liabilities_assets_ratio',
    'revenue_growth_3yr','asset_growth_3yr','revenue_volatility_3yr','board_independence'] if c in p]
profile=p.groupby('validation_signal_pattern',dropna=False).agg(organizations=('ein','size'))
for c in compare_cols:
    profile[c+'_median']=p.groupby('validation_signal_pattern')[c].median()
    profile[c+'_missing_pct']=100*p[c].isna().groupby(p.validation_signal_pattern).mean()
profile=profile.reset_index()
display(profile)
profile.to_csv(OUT/'07_signal_pattern_profiles.csv',index=False)
print('No independently confirmed fraud outcome is available in these outputs.')

## 8. Investigator review rubric and blinded sampling plan

Create a **review protocol** and a stratified random sample for potential human assessment. This notebook does not expose names, make allegations, or assign criminality scores. The sample file includes only randomized review IDs, signal strata, and a salted-for-this-run lookup file kept **locally** for authorized analysts. The lookup is sensitive because it maps anonymous IDs to EINs; control access and do not distribute it with external reports.

Recommended rubric: (1) source-document completeness; (2) accounting plausibility and explanation; (3) governance context; (4) cross-year consistency; (5) corroborating independently sourced evidence; (6) recommended disposition: no follow-up / request additional records / analyst escalation. Record evidence and uncertainty, not subjective fraud likelihood.

In [ ]:
rng=np.random.default_rng(13013)
# Keep the review sample small and balanced across distinct signal patterns.
sample_parts=[]
for group in ['both','supervised_only','peer_only','neither']:
    sub=p.loc[p.validation_signal_pattern.eq(group),['ein','OBJECTID','validation_signal_pattern']]
    n=min(25,len(sub))
    if n:
        sample_parts.append(sub.sample(n=n,random_state=13013))
review=pd.concat(sample_parts,ignore_index=True)
review=review.iloc[rng.permutation(len(review))].reset_index(drop=True)
review.insert(0,'review_id',[f'R{j:04d}' for j in range(1,len(review)+1)])
lookup=review[['review_id','ein','OBJECTID','validation_signal_pattern']].copy()
# Only blinded review IDs in the reviewer worksheet. Strata are retained in a separate lookup for analysis.
worksheet=review[['review_id']].copy()
for field in ['source_documents_reviewed','financial_anomaly_explanation','governance_context',
              'historical_consistency','independent_corroboration','data_quality_concerns',
              'review_disposition','reviewer_notes']:
    worksheet[field]=''
worksheet.to_csv(OUT/'08_blinded_investigator_review_template.csv',index=False)
lookup.to_csv(OUT/'08_RESTRICTED_review_id_lookup.csv',index=False)
review_protocol={
 'purpose':'Assess research-lead utility, not adjudicate fraud',
 'sampling':'Up to 25 per signal stratum, random seed 13013; unranked comparison from remaining primary organizations',
 'reviewer_blinding':'Reviewers receive review IDs only; lookup file is restricted',
 'review_dimensions':['source document completeness','accounting plausibility','governance context',
                      'cross-year consistency','independent corroboration','data quality'],
 'dispositions':['no follow-up','request additional records','analyst escalation'],
 'rules':['Do not use a score as proof of fraud','Record documentary basis for each assessment',
          'Use authorized data and access controls','Do not publish individual EINs or allegations',
          'Consider two reviewers for a subset to assess consistency']}
(OUT/'08_review_protocol.json').write_text(json.dumps(review_protocol,indent=2))
print('Blinded review cases:',len(worksheet))
print('Restricted ID lookup saved locally; do not distribute with public reports.')

## 9. Decision gates, reproducibility, and validation summary

A successful notebook run is **not** a deployment approval. The following gates are intentionally conservative. A historical-export discrepancy requires fixing Notebook 12 before using its trends. Ranking stability and investigator usefulness require explicit human judgment and stronger outcomes.

In [ ]:
qa={
 'primary_organizations':int(len(p)),
 'limited_data_organizations':int(len(l)),
 'total_unique_eins':int(len(p)+len(l)),
 'duplicate_eins_across_queues':int(len(set(p.ein)&set(l.ein))),
 'historical_source_rows_2022_2024':int(len(hist)),
 'historical_duplicate_ein_year_groups':int(len(dups)),
 'source_three_year_coverage_organizations':int(check.source_has_all_3_years.sum()),
 'notebook12_history_export_discrepancy':bool(upstream_history_mismatch),
 'top1000_supervised_peer_overlap':int(agreement.loc[agreement.top_k.eq(1000),'supervised_peer_overlap'].iloc[0]),
 'review_cases_generated':int(len(worksheet)),
 'fraud_detection_validated':False,
 'ready_for_operational_use':False
}
(OUT/'09_validation_summary.json').write_text(json.dumps(qa,indent=2))
manifest={
 'notebook':'13 — Investigation Queue Validation and Investigative Usefulness',
 'inputs':{'primary':str(IN12_PRIMARY),'limited':str(IN12_LIMITED),'features':str(FEATURES)},
 'tax_year':2024,'random_seed':13013,
 'weight_sensitivity':'Alternative supervised/peer percentile blends, compared to unchanged Notebook 12 consensus',
 'peer_test':'Revenue-group concentration plus global-percentile proxy; not full peer-group re-scoring',
 'history':'Normalized EIN in source; coverage check does not resolve amended or duplicate filings',
 'privacy':'Restricted review lookup maps review IDs to EINs and must not be distributed',
 'limitations':['No confirmed fraud labels','No untouched holdout for 2023-2024',
   'No point-in-time availability validation','No investigator review outcomes yet',
   'Proxy peer comparison is not alternative full anomaly scoring',
   'Descriptive outliers are not evidence of wrongdoing']
}
(OUT/'09_manifest.json').write_text(json.dumps(manifest,indent=2))
display(pd.DataFrame([qa]).T.rename(columns={0:'value'}))
print('Saved results to:',OUT)
if upstream_history_mismatch:
    print('ACTION REQUIRED: Fix historical join in Notebook 12 and regenerate queue before relying on trend fields.')
print('No fraud-detection performance or operational readiness is claimed.')

## Final Results, Interpretation, and Next Steps

### 1. Overview

Notebook 13 evaluated the stability, completeness, and potential investigative usefulness of the organization-level research queue created in Notebook 12.

The analysis examined historical data coverage, financial data quality, ranking sensitivity, peer-group effects, differences between risk-identification methods, and the feasibility of structured investigator review.

The objective was to assess the reliability of the analytical prioritization process, not to establish fraud-detection accuracy.

### 2. Historical Data Validation

The historical EIN join issue identified during the initial validation run was corrected in Notebook 12.

The final validation confirmed:

| Tax year | Revenue available | Assets available |
|---|---:|---:|
| 2022 | 332,460 | 312,243 |
| 2023 | 343,282 | 330,029 |
| 2024 | 355,973 | 355,973 |

Additional findings:

- 1,031,943 historical filing records were matched.
- 330,550 organizations had source filings in all three years.
- No duplicate historical EIN/year combinations were detected.
- No historical export discrepancies remained.

**Conclusion:** The historical join and export issue is resolved. Historical financial data is available where supported by the source records.

### 3. Investigation Queue Integrity

The final queue contains:

| Population | Organizations |
|---|---:|
| Primary review | 329,440 |
| Limited-data | 26,533 |
| **Total** | **355,973** |

No duplicate EINs were detected across the queues.

The investigation queue passed its structural integrity checks.

### 4. Ranking Stability and Weight Sensitivity

The exploratory combined ranking was tested using alternative supervised-model weights.

| Supervised weight | Top 100 overlap | Top 500 overlap | Top 1,000 overlap |
|---|---:|---:|---:|
| 25% | 72 | 403 | 815 |
| 50% | 100 | 500 | 1,000 |
| 75% | 80 | 384 | 825 |

These results demonstrate that changing the relative weights materially changes which organizations appear in the highest-priority groups.

For example, increasing the supervised weight to 75% replaces 116 organizations in the top 500.

**Conclusion:** The combined ranking is sensitive to methodological choices. Equal weighting remains an exploratory baseline rather than a validated optimal approach.

### 5. Differences Between Supervised and Peer-Anomaly Rankings

Only 32 organizations appeared in the top 1,000 of both approaches.

The methods also prioritized organizations of substantially different sizes.

| Signal classification | Organizations | Median annual revenue |
|---|---:|---:|
| Supervised only | 968 | $407,274,665 |
| Peer anomalies only | 968 | $649,912 |
| Both approaches | 32 | $474,459 |
| Neither approach | 327,472 | $541,854 |

The supervised-only group had a dramatically higher median revenue than the peer-only group.

This raises an important question about whether the supervised model's ranking is disproportionately influenced by organization size.

However, these descriptive comparisons do not establish that size is causing the model's predictions.

**Conclusion:** Organization-size effects require further investigation before the supervised ranking can be considered suitable for operational prioritization.

### 6. Peer-Group Sensitivity

The analysis found that peer-group definitions influence which organizations appear most anomalous.

Only 27 organizations in the top 100 of the existing peer-anomaly ranking also appeared in the top 100 of the global comparison proxy.

This suggests that comparing nonprofits with similarly sized organizations produces substantially different results from comparing them against the overall population.

The global comparison was a diagnostic proxy, not a complete alternative anomaly model.

**Conclusion:** Peer-group design is a material modeling decision. Future work should investigate whether nonprofit subsector and mission improve the relevance of peer comparisons.

### 7. Extreme Values and Data Quality

The analysis examined whether the highest-ranked organizations contained unusually large financial values or incomplete information.

The top-ranked organizations generally retained strong indicator coverage, but extreme financial values were present.

These observations require additional investigation because extreme values may reflect legitimate organizational characteristics, accounting practices, or reporting errors.

**Conclusion:** High anomaly scores should be accompanied by checks for data quality and plausible non-fraud explanations.

### 8. Investigator Review Framework

Notebook 13 generated:

- A blinded, stratified sample of 100 organizations.
- A separate restricted identifier lookup.
- A structured investigative review rubric.

This provides a framework for evaluating whether highly ranked organizations present meaningful investigative leads.

However, no investigator assessments have yet been completed.

Therefore, the project cannot currently measure investigative usefulness, confirmed fraud precision, or operational effectiveness.

### 9. Remaining Limitations

The principal limitations are:

1. No independently confirmed fraud outcomes are available for model validation.
2. The supervised model was trained on self-reported significant asset diversions.
3. The combined ranking weights remain exploratory.
4. Organization size may materially influence supervised rankings.
5. Peer-group definitions affect anomaly results.
6. Extreme financial values may have legitimate explanations.
7. Investigator review has not yet been conducted.
8. The current results do not establish prospective investigative performance.

### 10. Next Steps: Notebook 14 — Organization Size Bias and Model Robustness

Notebook 14 should investigate whether the supervised model is disproportionately prioritizing large nonprofits.

The next analysis should:

1. Examine supervised model scores across revenue bands.
2. Compare ranking performance within organizations of similar size.
3. Evaluate relationships between revenue, assets, and model scores.
4. Investigate whether size-adjusted rankings change the highest-priority organizations.
5. Compare the financial characteristics of organizations selected by supervised and peer-anomaly methods.
6. Determine whether organization-size effects can be mitigated without compromising useful predictive information.

### 11. Final Conclusion

Notebook 13 confirmed that the historical data pipeline is functioning correctly and that the organization-level investigation queue passes its structural validation checks.

It also identified two important methodological concerns: sensitivity to ranking weights and substantial differences in organization size between supervised and peer-anomaly selections.

**Decision:** Retain Notebooks 12 and 13 as completed analytical phases. Proceed to Notebook 14 to investigate organization-size effects and model robustness before considering operational use.

The current rankings should continue to be treated as exploratory investigative leads, not validated fraud predictions or evidence of criminal activity.